# FinSight AI · Research Part 1: Grounding benchmarks

This notebook measures the three components that decide whether a personal finance assistant gives correct, traceable answers. Every experiment uses **real public data**, with no curated or synthetic examples:

| Exp | Component in FinSight | Real dataset | Question |
|---|---|---|---|
| **E1** | Question router (`ai_context.detect_intent`) | **Banking77**: 13,083 real online-banking customer queries, 77 intents | Does a learned router beat FinSight's lexical approach, with full data and with only 5 or 10 examples per intent? |
| **E2** | Knowledge retrieval (`knowledge.py`, 256-dim hashed bag-of-words) | **FiQA-2018** (BEIR): 57,638 real finance forum answers, 648 test questions | How much do BM25, dense embeddings, hybrid search and reranking improve on FinSight's current retriever? |
| **E3** | Numeric grounding ("proof on every number") | **FinQA**: 1,147 test questions over real S&P 500 earnings reports | Does making the LLM write a program and checking that every number in it comes from the source beat answering directly? |

**How to run on Kaggle**
1. *Settings → Accelerator → GPU T4 x2* (or P100).
2. *Settings → Internet → On* (needs a phone-verified Kaggle account).
3. *Run all*. On a T4, a full run takes roughly 1–1.5 hours; set `FAST = True` below for a 10-minute smoke test.

Results are written to `/kaggle/working/part1_results/` (`summary.csv`, `results.json` and per-item files). Download them from the *Output* panel.

## 0. Configuration

In [ ]:
FAST = False          # True: small samples for a quick end-to-end check
SEED = 13

RUN_E1_ROUTING   = True
RUN_E2_RETRIEVAL = True
RUN_E3_NUMERIC   = True

# Models (all open, downloaded from Hugging Face)
EMBED_MODELS = {
    "bge-small-en-v1.5": dict(name="BAAI/bge-small-en-v1.5",
                              query_prefix="Represent this sentence for searching relevant passages: "),
    "bge-m3":            dict(name="BAAI/bge-m3", query_prefix=""),
}
RERANKER_MODEL = "BAAI/bge-reranker-base"
LLM_NAME       = "Qwen/Qwen2.5-3B-Instruct"   # try "Qwen/Qwen2.5-7B-Instruct" with LLM_4BIT = True
LLM_4BIT       = False

FINQA_N        = 120 if FAST else 400   # number of FinQA test questions (max 1147)
FIQA_QUERIES   = 100 if FAST else None  # None = all 648 test queries
FIQA_CORPUS_SAMPLE = 6000 if FAST else None  # None = full 57,638-document corpus
GEN_BATCH      = 8
MAX_NEW_TOKENS = 384

## 1. Setup

In [ ]:
import subprocess, sys, importlib

def ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg], check=False)

for pkg, pip_name in [("sentence_transformers", "sentence-transformers"), ("transformers", None),
                      ("sklearn", "scikit-learn"), ("scipy", None), ("pandas", None), ("matplotlib", None)]:
    ensure(pkg, pip_name)
if LLM_4BIT:
    ensure("bitsandbytes")

In [ ]:
import os, re, io, json, math, time, random, zipfile, hashlib, signal, urllib.request, platform
from collections import Counter, defaultdict
from decimal import Decimal
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

random.seed(SEED); np.random.seed(SEED)

OUT = "/kaggle/working/part1_results" if os.path.isdir("/kaggle/working") else os.path.abspath("part1_results")
CACHE = os.path.join(OUT, "cache")
os.makedirs(CACHE, exist_ok=True)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(SEED)
except ImportError:
    torch, DEVICE = None, "cpu"

try:
    import sentence_transformers  # noqa: F401
    HAVE_ST = True
except ImportError:
    HAVE_ST = False

RESULTS = {"config": dict(FAST=FAST, SEED=SEED, LLM_NAME=LLM_NAME, LLM_4BIT=LLM_4BIT, FINQA_N=FINQA_N,
                          EMBED_MODELS={k: v["name"] for k, v in EMBED_MODELS.items()},
                          RERANKER_MODEL=RERANKER_MODEL),
           "env": dict(python=platform.python_version(), device=DEVICE,
                       gpu=(torch.cuda.get_device_name(0) if torch is not None and DEVICE == "cuda" else None),
                       torch=getattr(torch, "__version__", None)),
           "E1": {}, "E2": {}, "E3": {}}
SUMMARY_ROWS = []
print(RESULTS["env"], "| sentence-transformers:", HAVE_ST, "| output:", OUT)

def download(url, name):
    path = os.path.join(CACHE, name)
    if not os.path.exists(path):
        print("downloading", url)
        req = urllib.request.Request(url, headers={"User-Agent": "finsight-research"})
        with urllib.request.urlopen(req, timeout=300) as r, open(path, "wb") as f:
            f.write(r.read())
    return path

def bootstrap_ci(values, n=1000, seed=SEED):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), size=(n, len(values)))].mean(axis=1)
    return (float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5)))

def add_summary(exp, method, metric, value, ci=None, **extra):
    row = dict(experiment=exp, method=method, metric=metric, value=round(float(value), 4))
    if ci is not None:
        row["ci_low"], row["ci_high"] = round(ci[0], 4), round(ci[1], 4)
    row.update(extra)
    SUMMARY_ROWS.append(row)

def save_all():
    pd.DataFrame(SUMMARY_ROWS).to_csv(os.path.join(OUT, "summary.csv"), index=False)
    with open(os.path.join(OUT, "results.json"), "w") as f:
        json.dump(RESULTS, f, indent=2, default=str)

### FinSight's current text representation (ported exactly from `backend/app/services/knowledge.py`)

256-dimensional signed feature hashing over unigrams and bigrams with log term weights, L2-normalised, plus a keyword-overlap boost of 0.35 at retrieval time. This is the baseline that E1 and E2 try to beat.

In [ ]:
FS_DIMENSIONS = 256
FS_TOKEN = re.compile(r"[a-z][a-z0-9']+")
FS_STOP = {
    "a", "an", "and", "are", "as", "at", "be", "by", "can", "do", "does", "for", "from",
    "how", "i", "if", "in", "is", "it", "me", "my", "of", "on", "or", "should", "so",
    "that", "the", "their", "them", "this", "to", "was", "we", "what", "when", "where",
    "which", "who", "why", "will", "with", "you", "your",
}

def fs_tokenize(text):
    return [t for t in FS_TOKEN.findall(text.lower()) if t not in FS_STOP]

def fs_embed(text):
    values = np.zeros(FS_DIMENSIONS, dtype=np.float32)
    tokens = fs_tokenize(text)
    if not tokens:
        return values
    feats = list(tokens) + [f"{tokens[i]} {tokens[i + 1]}" for i in range(len(tokens) - 1)]
    for feat, count in Counter(feats).items():
        digest = hashlib.blake2b(feat.encode("utf-8"), digest_size=4).digest()
        bucket = int.from_bytes(digest, "big") % FS_DIMENSIONS
        sign = 1.0 if digest[0] % 2 == 0 else -1.0
        values[bucket] += sign * (1.0 + math.log(count))
    norm = np.linalg.norm(values)
    return values / norm if norm else values

def fs_embed_many(texts):
    return np.vstack([fs_embed(t) for t in texts])

### Shared embedding helper

In [ ]:
_ST_CACHE = {}

def st_model(name):
    from sentence_transformers import SentenceTransformer
    if name not in _ST_CACHE:
        m = SentenceTransformer(name, device=DEVICE)
        m.max_seq_length = min(getattr(m, "max_seq_length", 512) or 512, 384)
        if DEVICE == "cuda":
            m.half()
        _ST_CACHE[name] = m
    return _ST_CACHE[name]

def st_encode(name, texts, prefix="", batch_size=64):
    model = st_model(name)
    return model.encode([prefix + t for t in texts], batch_size=batch_size, normalize_embeddings=True,
                        show_progress_bar=len(texts) > 2000, convert_to_numpy=True).astype(np.float32)

def free_gpu():
    _ST_CACHE.clear()
    import gc; gc.collect()
    if torch is not None and DEVICE == "cuda":
        torch.cuda.empty_cache()

## E1 · Question routing on Banking77

FinSight routes each question to an intent with hand-written regex rules. Regex rules cannot be written for 77 real-world intents in a fair, reproducible way, so the lexical baseline here is **FinSight's own hashed representation with a nearest-intent-centroid rule**: the same text features the app already uses, with no learning beyond averaging.

Two data regimes are reported: **full** training data (10,003 queries) and **few-shot** (5 and 10 examples per intent, averaged over 3 random draws). Few-shot matters because a new FinSight intent starts with a handful of examples. Metrics: accuracy (with bootstrap 95% CI), macro-F1, and encoding plus prediction time per query.

In [ ]:
if RUN_E1_ROUTING:
    from sklearn.linear_model import LogisticRegression
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.pipeline import make_pipeline, make_union
    from sklearn.metrics import accuracy_score, f1_score

    base = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/"
    b77_train = pd.read_csv(download(base + "train.csv", "banking77_train.csv"))
    b77_test = pd.read_csv(download(base + "test.csv", "banking77_test.csv"))
    if FAST:
        b77_test = b77_test.sample(800, random_state=SEED).reset_index(drop=True)
    labels = sorted(b77_train["category"].unique())
    lab2id = {l: i for i, l in enumerate(labels)}
    ytr_all = b77_train["category"].map(lab2id).to_numpy()
    yte = b77_test["category"].map(lab2id).to_numpy()
    print(f"Banking77: {len(b77_train)} train, {len(b77_test)} test, {len(labels)} intents")

    def few_shot_indices(k, seed):
        rng = np.random.default_rng(seed)
        idx = []
        for c in range(len(labels)):
            pool = np.flatnonzero(ytr_all == c)
            idx.extend(rng.choice(pool, size=min(k, len(pool)), replace=False))
        return np.array(sorted(idx))

    regimes = [("full", [np.arange(len(b77_train))])] + \
              [(f"{k}-shot", [few_shot_indices(k, SEED + s) for s in range(3)]) for k in (5, 10)]

    # Pre-compute features once; each method indexes into them.
    feats = {}
    t = time.perf_counter(); feats["finsight-hash"] = (fs_embed_many(b77_train["text"]), fs_embed_many(b77_test["text"]))
    hash_ms = (time.perf_counter() - t) / (len(b77_train) + len(b77_test)) * 1000
    enc_ms = {"finsight-hash": hash_ms}
    if HAVE_ST:
        for key, spec in EMBED_MODELS.items():
            t = time.perf_counter()
            feats[key] = (st_encode(spec["name"], b77_train["text"].tolist()), st_encode(spec["name"], b77_test["text"].tolist()))
            enc_ms[key] = (time.perf_counter() - t) / (len(b77_train) + len(b77_test)) * 1000
        free_gpu()

    def centroid_predict(Xtr, ytr, Xte):
        C = np.vstack([Xtr[ytr == c].mean(axis=0) for c in range(len(labels))])
        C /= np.linalg.norm(C, axis=1, keepdims=True) + 1e-12
        return (Xte @ C.T).argmax(axis=1)

    def logreg_predict(Xtr, ytr, Xte):
        clf = LogisticRegression(max_iter=2000, C=10.0)
        clf.fit(Xtr, ytr)
        return clf.predict(Xte)

    def tfidf_predict(idx):
        vec = make_union(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=1),
                         TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True, min_df=1))
        pipe = make_pipeline(vec, LogisticRegression(max_iter=2000, C=10.0))
        pipe.fit(b77_train["text"].iloc[idx], ytr_all[idx])
        return pipe.predict(b77_test["text"])

    methods = [("FinSight hashed BoW + centroid (current)", "finsight-hash", "centroid"),
               ("FinSight hashed BoW + logistic regression", "finsight-hash", "logreg"),
               ("TF-IDF word+char + logistic regression", None, "tfidf")]
    for key in EMBED_MODELS:
        if key in feats:
            methods += [(f"{key} + centroid", key, "centroid"), (f"{key} + logistic regression", key, "logreg")]

    e1_rows, e1_items = [], {}
    for regime, draws in regimes:
        for label, fkey, kind in methods:
            accs, f1s, correct_vecs, secs = [], [], [], []
            for idx in draws:
                t = time.perf_counter()
                if kind == "tfidf":
                    pred = tfidf_predict(idx)
                else:
                    Xtr, Xte = feats[fkey]
                    pred = (centroid_predict if kind == "centroid" else logreg_predict)(Xtr[idx], ytr_all[idx], Xte)
                secs.append(time.perf_counter() - t)
                accs.append(accuracy_score(yte, pred)); f1s.append(f1_score(yte, pred, average="macro"))
                correct_vecs.append((pred == yte).astype(float))
            correct = np.mean(correct_vecs, axis=0)
            ci = bootstrap_ci(correct)
            ms = (np.mean(secs) / len(yte)) * 1000 + enc_ms.get(fkey, 0.0)
            e1_rows.append(dict(regime=regime, method=label, accuracy=np.mean(accs), acc_std=np.std(accs),
                                ci_low=ci[0], ci_high=ci[1], macro_f1=np.mean(f1s), ms_per_query=ms))
            add_summary("E1 routing", f"{label} [{regime}]", "accuracy", np.mean(accs), ci, macro_f1=round(float(np.mean(f1s)), 4))
            print(f"{regime:7s} | {label:45s} | acc {np.mean(accs):.4f} ± {np.std(accs):.4f} | F1 {np.mean(f1s):.4f}")
    e1 = pd.DataFrame(e1_rows)
    e1.to_csv(os.path.join(OUT, "e1_routing.csv"), index=False)
    RESULTS["E1"] = e1.to_dict(orient="records")
    save_all()
    display(e1.pivot(index="method", columns="regime", values="accuracy").round(4)) if "display" in dir() else print(e1)

## E2 · Knowledge retrieval on FiQA-2018 (BEIR)

FinSight retrieves knowledge with 256-dim hashed bag-of-words plus a 0.35 keyword-overlap boost (`knowledge.retrieve_knowledge`). We compare it on the standard BEIR FiQA split against:

- **BM25** (k1 = 1.2, b = 0.75, the BEIR Elasticsearch defaults; the BEIR paper reports nDCG@10 = 0.236 for BM25 on FiQA)
- **Dense** retrieval with `bge-small-en-v1.5` and the multilingual `bge-m3` (which also covers Hindi, needed later for Hinglish)
- **Hybrid**: reciprocal-rank fusion (k = 60) of BM25 and the best dense model
- **Hybrid + rerank**: a cross-encoder reranks the hybrid top 100

Metrics: nDCG@10 (primary, as in BEIR), MRR@10, Recall@100, and query latency.

In [ ]:
def load_fiqa():
    """BEIR FiQA from the official BEIR mirror, falling back to the MTEB copy on Hugging Face."""
    try:
        zpath = download("https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip", "fiqa.zip")
        with zipfile.ZipFile(zpath) as z:
            corpus = {}
            with z.open("fiqa/corpus.jsonl") as f:
                for line in io.TextIOWrapper(f, encoding="utf-8"):
                    d = json.loads(line); corpus[str(d["_id"])] = (d.get("title", "") + " " + d["text"]).strip()
            queries = {}
            with z.open("fiqa/queries.jsonl") as f:
                for line in io.TextIOWrapper(f, encoding="utf-8"):
                    d = json.loads(line); queries[str(d["_id"])] = d["text"]
            qrels = defaultdict(dict)
            with z.open("fiqa/qrels/test.tsv") as f:
                for i, line in enumerate(io.TextIOWrapper(f, encoding="utf-8")):
                    if i == 0:
                        continue
                    q, d, s = line.rstrip("\n").split("\t")
                    if int(s) > 0:
                        qrels[q][d] = int(s)
        return corpus, queries, dict(qrels)
    except Exception as exc:
        print("BEIR mirror failed, using Hugging Face mteb/fiqa:", exc)
        from datasets import load_dataset
        c = load_dataset("mteb/fiqa", "corpus", split="corpus")
        qd = load_dataset("mteb/fiqa", "queries", split="queries")
        r = load_dataset("mteb/fiqa", "default", split="test")
        corpus = {str(x["_id"]): (x.get("title", "") + " " + x["text"]).strip() for x in c}
        queries = {str(x["_id"]): x["text"] for x in qd}
        qrels = defaultdict(dict)
        for x in r:
            if int(x["score"]) > 0:
                qrels[str(x["query-id"])][str(x["corpus-id"])] = int(x["score"])
        return corpus, queries, dict(qrels)

BM25_TOKEN = re.compile(r"[a-z0-9]+")

def bm25_tokenize(text):
    return BM25_TOKEN.findall(text.lower())

class SparseBM25:
    def __init__(self, docs, k1=1.2, b=0.75):
        from sklearn.feature_extraction.text import CountVectorizer
        self.vec = CountVectorizer(tokenizer=bm25_tokenize, lowercase=False, token_pattern=None)
        tf = self.vec.fit_transform(docs).tocsc().astype(np.float32)
        n = tf.shape[0]
        dl = np.asarray(tf.sum(axis=1)).ravel()
        df = np.diff(tf.indptr)
        self.idf = np.log(1 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)
        tf = tf.tocsr()
        denom = k1 * (1 - b + b * dl / dl.mean())
        rows = np.repeat(np.arange(n), np.diff(tf.indptr))
        data = tf.data * (k1 + 1) / (tf.data + denom[rows])
        self.W = sp.csr_matrix((data, tf.indices, tf.indptr), shape=tf.shape).tocsc()

    def scores(self, queries):
        Q = self.vec.transform(queries)
        Q.data[:] = 1.0
        Q = Q.multiply(self.idf).tocsr()
        return np.asarray((Q @ self.W.T).todense(), dtype=np.float32)

def finsight_retrieval_scores(q_texts, doc_texts, doc_hash):
    """Exact port of retrieve_knowledge scoring: cosine(hash) + 0.35 * keyword overlap."""
    from sklearn.feature_extraction.text import CountVectorizer
    vec = CountVectorizer(tokenizer=lambda s: sorted(set(fs_tokenize(s))), lowercase=False, token_pattern=None, binary=True)
    D = vec.fit_transform(doc_texts).astype(np.float32)
    Qb = vec.transform(q_texts).astype(np.float32)
    q_len = np.array([max(len(set(fs_tokenize(q))), 1) for q in q_texts], dtype=np.float32)[:, None]
    overlap = np.asarray((Qb @ D.T).todense()) / q_len
    sim = fs_embed_many(q_texts) @ doc_hash.T
    return sim + 0.35 * overlap

def rank_from_scores(S, doc_ids, k=100):
    top = np.argpartition(-S, kth=min(k, S.shape[1] - 1), axis=1)[:, :k]
    out = []
    for i in range(S.shape[0]):
        order = top[i][np.argsort(-S[i, top[i]])]
        out.append([doc_ids[j] for j in order])
    return out

def rrf(*rankings, k=60, depth=100):
    fused = []
    for lists in zip(*rankings):
        score = defaultdict(float)
        for lst in lists:
            for r, d in enumerate(lst):
                score[d] += 1.0 / (k + r + 1)
        fused.append([d for d, _ in sorted(score.items(), key=lambda x: -x[1])[:depth]])
    return fused

def ir_metrics(runs, qids, qrels):
    nd, mrr, rec = [], [], []
    for qid, ranked in zip(qids, runs):
        rel = qrels.get(qid, {})
        gains = [rel.get(d, 0) for d in ranked[:10]]
        dcg = sum(g / math.log2(i + 2) for i, g in enumerate(gains))
        ideal = sorted(rel.values(), reverse=True)[:10]
        idcg = sum(g / math.log2(i + 2) for i, g in enumerate(ideal))
        nd.append(dcg / idcg if idcg else 0.0)
        rr = 0.0
        for i, d in enumerate(ranked[:10]):
            if d in rel:
                rr = 1.0 / (i + 1); break
        mrr.append(rr)
        rec.append(len(set(ranked[:100]) & set(rel)) / len(rel) if rel else 0.0)
    return np.array(nd), np.array(mrr), np.array(rec)

In [ ]:
if RUN_E2_RETRIEVAL:
    corpus, queries, qrels = load_fiqa()
    qids = [q for q in qrels if q in queries]
    if FIQA_QUERIES:
        qids = random.Random(SEED).sample(qids, min(FIQA_QUERIES, len(qids)))
    if FIQA_CORPUS_SAMPLE:
        keep = {d for q in qids for d in qrels[q]}
        others = [d for d in corpus if d not in keep]
        keep |= set(random.Random(SEED).sample(others, max(0, FIQA_CORPUS_SAMPLE - len(keep))))
        corpus = {d: corpus[d] for d in corpus if d in keep}
    doc_ids = list(corpus)
    doc_texts = [corpus[d] for d in doc_ids]
    q_texts = [queries[q] for q in qids]
    print(f"FiQA: {len(doc_ids)} documents, {len(qids)} test queries")

    runs, latency = {}, {}

    t = time.perf_counter(); doc_hash = fs_embed_many(doc_texts); t_index = time.perf_counter() - t
    t = time.perf_counter(); S = finsight_retrieval_scores(q_texts, doc_texts, doc_hash)
    runs["FinSight hashed BoW + overlap (current)"] = rank_from_scores(S, doc_ids)
    latency["FinSight hashed BoW + overlap (current)"] = (time.perf_counter() - t) / len(qids) * 1000
    del S

    bm25 = SparseBM25(doc_texts)
    t = time.perf_counter(); S = bm25.scores(q_texts)
    runs["BM25"] = rank_from_scores(S, doc_ids)
    latency["BM25"] = (time.perf_counter() - t) / len(qids) * 1000
    del S

    best_dense = None
    if HAVE_ST:
        for key, spec in EMBED_MODELS.items():
            D = st_encode(spec["name"], doc_texts, batch_size=64)
            t = time.perf_counter()
            Qe = st_encode(spec["name"], q_texts, prefix=spec["query_prefix"])
            runs[f"Dense {key}"] = rank_from_scores(Qe @ D.T, doc_ids)
            latency[f"Dense {key}"] = (time.perf_counter() - t) / len(qids) * 1000
            del D
            free_gpu()
        dense_names = [n for n in runs if n.startswith("Dense ")]
        best_dense = max(dense_names, key=lambda n: ir_metrics(runs[n], qids, qrels)[0].mean())
        runs[f"Hybrid RRF (BM25 + {best_dense[6:]})"] = rrf(runs["BM25"], runs[best_dense])

        from sentence_transformers import CrossEncoder
        ce = CrossEncoder(RERANKER_MODEL, device=DEVICE, max_length=512)
        hybrid = runs[f"Hybrid RRF (BM25 + {best_dense[6:]})"]
        reranked = []
        t = time.perf_counter()
        for q, cand in zip(q_texts, hybrid):
            scores = ce.predict([(q, corpus[d]) for d in cand], batch_size=64, show_progress_bar=False)
            reranked.append([cand[i] for i in np.argsort(-np.asarray(scores))])
        latency[f"Hybrid + rerank ({RERANKER_MODEL.split('/')[-1]})"] = (time.perf_counter() - t) / len(qids) * 1000
        runs[f"Hybrid + rerank ({RERANKER_MODEL.split('/')[-1]})"] = reranked
        del ce; free_gpu()

    e2_rows = []
    for name, run in runs.items():
        nd, mrr, rec = ir_metrics(run, qids, qrels)
        ci = bootstrap_ci(nd)
        e2_rows.append(dict(method=name, ndcg10=nd.mean(), ci_low=ci[0], ci_high=ci[1], mrr10=mrr.mean(),
                            recall100=rec.mean(), ms_per_query=latency.get(name)))
        add_summary("E2 retrieval", name, "nDCG@10", nd.mean(), ci, mrr10=round(float(mrr.mean()), 4), recall100=round(float(rec.mean()), 4))
        print(f"{name:50s} nDCG@10 {nd.mean():.4f}  MRR@10 {mrr.mean():.4f}  R@100 {rec.mean():.4f}")
    e2 = pd.DataFrame(e2_rows)
    e2.to_csv(os.path.join(OUT, "e2_retrieval.csv"), index=False)
    RESULTS["E2"] = dict(n_docs=len(doc_ids), n_queries=len(qids), rows=e2.to_dict(orient="records"))
    save_all()

## E3 · Numeric grounding on FinQA

The research hypothesis behind "proof on every number": an LLM is more accurate and more trustworthy when it **writes a program over numbers taken from the source** and a deterministic checker verifies those numbers, than when it states an answer directly.

Arms (same model, same prompts except the answer format, greedy decoding):

- **A · Direct**: the model states the final number.
- **B · Program**: the model writes Python that stores the answer in `ans`; we execute it in a sandbox.
- **C · Program + verifier**: like B, but every numeric constant in the program must appear in the report (after normalising commas, `$`, `%` and parentheses). If a constant is ungrounded or the program fails, the model gets one retry with the problem stated; if it still fails, the system **abstains**.

Metrics: accuracy over all questions, coverage (share answered), selective accuracy (accuracy when answering), and the **ungrounded-program rate** (share of programs using at least one number not in the source). Scoring follows FinQA execution accuracy with a 1% relative tolerance, and also accepts a percentage given as a fraction or the reverse (for example 0.12 vs 12%). This leniency is applied identically to all arms.

In [ ]:
NUM_RE = re.compile(r"\(?-?\$?\d[\d,]*\.?\d*%?\)?")
TRIVIAL_CONSTANTS = {0.0, 1.0, 2.0, 3.0, 4.0, 10.0, 12.0, 100.0, 1000.0, 1e6, 1e9, 365.0}

def parse_number(tok):
    s = tok.strip()
    neg = s.startswith("(") and s.endswith(")")
    s = s.strip("()").replace("$", "").replace(",", "").rstrip("%")
    if s in {"", "-", "."}:
        return None
    try:
        v = float(s)
    except ValueError:
        return None
    return -v if neg else v

def table_to_text(table):
    return "\n".join(" | ".join(str(c).strip() for c in row) for row in table)

def finqa_context(item):
    return ("\n".join(item["pre_text"]) + "\n\nTable:\n" + table_to_text(item["table"]) + "\n\n" + "\n".join(item["post_text"])).strip()

def context_numbers(text):
    nums = set()
    for tok in NUM_RE.findall(text):
        v = parse_number(tok)
        if v is not None:
            nums.add(round(abs(v), 6))
    return nums

def is_grounded(value, nums):
    a = round(abs(value), 6)
    if a in TRIVIAL_CONSTANTS:
        return True
    return any(abs(a - n) <= 1e-9 + 1e-6 * max(a, n) for n in nums)

def program_constants(code):
    code = re.sub(r"#.*", "", code)
    out = []
    for tok in re.findall(r"(?<![\w.])\d[\d_]*\.?\d*(?:[eE][-+]?\d+)?", code):
        try:
            out.append(float(tok.replace("_", "")))
        except ValueError:
            pass
    return out

def ungrounded_constants(code, nums):
    return sorted({c for c in program_constants(code) if not is_grounded(c, nums)})

class _Timeout(Exception):
    pass

def run_program(code, seconds=3):
    safe_builtins = {"abs": abs, "min": min, "max": max, "sum": sum, "round": round, "len": len,
                     "float": float, "int": int, "range": range, "pow": pow, "sorted": sorted,
                     "bool": bool, "list": list, "dict": dict, "zip": zip, "enumerate": enumerate}
    env = {"__builtins__": safe_builtins, "math": math}
    if re.search(r"\b(import|open|exec|eval|__|os\.|sys\.)", code.replace("import math", "")):
        return None, "blocked: program uses a forbidden construct"
    def handler(signum, frame):
        raise _Timeout()
    old = signal.signal(signal.SIGALRM, handler) if hasattr(signal, "SIGALRM") else None
    try:
        if old is not None:
            signal.alarm(seconds)
        exec(code.replace("import math", ""), env)
        if "ans" not in env:
            return None, "the program did not set a variable named ans"
        return env["ans"], None
    except _Timeout:
        return None, "the program timed out"
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"
    finally:
        if old is not None:
            signal.alarm(0); signal.signal(signal.SIGALRM, old)

def extract_code(text):
    m = re.search(r"```(?:python)?\s*\n(.*?)```", text, flags=re.S)
    return (m.group(1) if m else text).strip()

def extract_direct_answer(text):
    t = text.strip().lower()
    m = re.search(r"answer\s*[:=]\s*(.+)", t)
    t = m.group(1) if m else t
    if re.match(r"^\W*(yes|no)\b", t):
        return re.match(r"^\W*(yes|no)\b", t).group(1)
    nums = [parse_number(x) for x in NUM_RE.findall(t)]
    nums = [n for n in nums if n is not None]
    return nums[-1] if nums else None

def normalise_pred(v):
    if isinstance(v, bool):
        return "yes" if v else "no"
    if isinstance(v, str):
        s = v.strip().lower()
        if s in {"yes", "no"}:
            return s
        return parse_number(s)
    if isinstance(v, (int, float, Decimal, np.floating, np.integer)):
        f = float(v)
        return f if math.isfinite(f) else None
    return None

def finqa_correct(pred, gold):
    pred = normalise_pred(pred)
    if pred is None:
        return False
    if isinstance(gold, str):
        g = gold.strip().lower()
        if g in {"yes", "no"}:
            return pred == g
        gold = parse_number(g)
        if gold is None:
            return False
    if isinstance(pred, str):
        return False
    gold = float(gold)
    for cand in (pred, pred * 100, pred / 100):
        if abs(cand - gold) <= max(1e-2 * abs(gold), 5e-3):
            return True
    return False

SYSTEM = "You are a careful financial analyst. Use only the information in the provided report excerpt."

def prompt_direct(item):
    return (f"Report excerpt:\n{finqa_context(item)}\n\nQuestion: {item['qa']['question']}\n\n"
            "Think briefly, then give the final answer on the last line as 'Answer: <number>' "
            "(or 'Answer: yes' / 'Answer: no'). Do not include units.")

def prompt_program(item):
    return (f"Report excerpt:\n{finqa_context(item)}\n\nQuestion: {item['qa']['question']}\n\n"
            "Write a short Python program that computes the answer and stores it in a variable named ans. "
            "Copy every number you need exactly as it appears in the report; do not invent or estimate values. "
            "For yes/no questions set ans to True or False. Return only the code in a ```python block.")

def prompt_retry(item, code, problem):
    return (prompt_program(item) + f"\n\nYour previous program was:\n```python\n{code}\n```\n"
            f"It was rejected: {problem}. Fix it and return only the corrected code.")

In [ ]:
_LLM = {}

def load_llm():
    if "model" in _LLM:
        return _LLM["tok"], _LLM["model"]
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kwargs = dict(device_map="auto")
    if LLM_4BIT:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    else:
        kwargs["torch_dtype"] = torch.float16 if DEVICE == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(LLM_NAME, **kwargs).eval()
    _LLM.update(tok=tok, model=model)
    return tok, model

def generate(prompts):
    tok, model = load_llm()
    outs = []
    for i in range(0, len(prompts), GEN_BATCH):
        chunk = prompts[i:i + GEN_BATCH]
        texts = [tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": p}],
                                         tokenize=False, add_generation_prompt=True) for p in chunk]
        enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=6000).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        outs.extend(tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True))
        print(f"  generated {min(i + GEN_BATCH, len(prompts))}/{len(prompts)}", end="\r")
    print()
    return outs

In [ ]:
if RUN_E3_NUMERIC:
    finqa = json.load(open(download("https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/test.json", "finqa_test.json")))
    items = random.Random(SEED).sample(finqa, min(FINQA_N, len(finqa)))
    golds = [it["qa"]["exe_ans"] for it in items]
    nums_per_item = [context_numbers(finqa_context(it)) for it in items]
    print(f"FinQA: {len(items)} of {len(finqa)} test questions")

    if torch is None:
        raise RuntimeError("E3 needs PyTorch and a GPU runtime")

    t0 = time.perf_counter()
    direct_raw = generate([prompt_direct(it) for it in items])
    t_direct = time.perf_counter() - t0
    direct_pred = [extract_direct_answer(x) for x in direct_raw]

    t0 = time.perf_counter()
    prog_raw = generate([prompt_program(it) for it in items])
    t_prog = time.perf_counter() - t0
    prog_code = [extract_code(x) for x in prog_raw]
    prog_exec = [run_program(c) for c in prog_code]
    prog_ungrounded = [ungrounded_constants(c, n) for c, n in zip(prog_code, nums_per_item)]

    # Arm C: verify, retry once with the problem stated, otherwise abstain.
    problems = []
    for (val, err), ung in zip(prog_exec, prog_ungrounded):
        if err:
            problems.append(err)
        elif normalise_pred(val) is None:
            problems.append("ans is not a number or True/False")
        elif ung:
            problems.append("these numbers do not appear in the report: " + ", ".join(f"{u:g}" for u in ung))
        else:
            problems.append(None)
    retry_idx = [i for i, p in enumerate(problems) if p]
    t0 = time.perf_counter()
    retry_raw = generate([prompt_retry(items[i], prog_code[i], problems[i]) for i in retry_idx]) if retry_idx else []
    t_retry = time.perf_counter() - t0
    ver_pred, ver_code, ver_status = [], list(prog_code), []
    retry_map = dict(zip(retry_idx, retry_raw))
    for i in range(len(items)):
        if problems[i] is None:
            ver_pred.append(prog_exec[i][0]); ver_status.append("verified first try"); continue
        code = extract_code(retry_map[i]); ver_code[i] = code
        val, err = run_program(code)
        if err or normalise_pred(val) is None or ungrounded_constants(code, nums_per_item[i]):
            ver_pred.append(None); ver_status.append("abstained")
        else:
            ver_pred.append(val); ver_status.append("verified after retry")

    def arm_stats(preds, answered_mask):
        correct = np.array([finqa_correct(p, g) if a else False for p, g, a in zip(preds, golds, answered_mask)], dtype=float)
        answered = np.array(answered_mask, dtype=bool)
        sel = correct[answered].mean() if answered.any() else float("nan")
        return correct, answered.mean(), sel

    arms = {
        "A · Direct answer": (direct_pred, [p is not None for p in direct_pred], None, t_direct),
        "B · Program": ([v for v, _ in prog_exec], [e is None for _, e in prog_exec],
                        np.mean([bool(u) for u in prog_ungrounded]), t_prog),
        "C · Program + verifier": (ver_pred, [p is not None for p in ver_pred],
                                   np.mean([bool(ungrounded_constants(c, n)) for c, n, p in zip(ver_code, nums_per_item, ver_pred) if p is not None] or [0.0]),
                                   t_prog + t_retry),
    }
    e3_rows = []
    for name, (preds, answered, ung_rate, secs) in arms.items():
        correct, coverage, sel = arm_stats(preds, answered)
        ci = bootstrap_ci(correct)
        e3_rows.append(dict(arm=name, accuracy=correct.mean(), ci_low=ci[0], ci_high=ci[1], coverage=coverage,
                            selective_accuracy=sel, ungrounded_program_rate=ung_rate,
                            sec_per_question=secs / len(items)))
        add_summary("E3 numeric", name, "accuracy", correct.mean(), ci, coverage=round(float(coverage), 4),
                    selective_accuracy=round(float(sel), 4))
        print(f"{name:26s} acc {correct.mean():.4f} [{ci[0]:.3f}, {ci[1]:.3f}]  coverage {coverage:.3f}  selective {sel:.4f}"
              + (f"  ungrounded {ung_rate:.3f}" if ung_rate is not None else ""))
    print("Arm C outcomes:", Counter(ver_status))
    e3 = pd.DataFrame(e3_rows)
    e3.to_csv(os.path.join(OUT, "e3_numeric.csv"), index=False)
    pd.DataFrame({"id": [it["id"] for it in items], "question": [it["qa"]["question"] for it in items],
                  "gold": golds, "direct_raw": direct_raw, "direct_pred": direct_pred,
                  "program": prog_code, "program_ans": [v for v, _ in prog_exec], "program_error": [e for _, e in prog_exec],
                  "ungrounded": prog_ungrounded, "verified_program": ver_code, "verified_ans": ver_pred,
                  "verifier_status": ver_status}).to_json(os.path.join(OUT, "e3_items.jsonl"), orient="records", lines=True)
    RESULTS["E3"] = dict(model=LLM_NAME, n=len(items), outcomes=dict(Counter(ver_status)), rows=e3.to_dict(orient="records"))
    save_all()

## Summary

`summary.csv` holds one row per method with its primary metric and 95% bootstrap CI. Compare each method to the row marked **(current)**: that is what FinSight runs today.

In [ ]:
summary = pd.DataFrame(SUMMARY_ROWS)
save_all()
print(summary.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
panels = [("E1 routing", "Routing accuracy (Banking77)"), ("E2 retrieval", "Retrieval nDCG@10 (FiQA)"), ("E3 numeric", f"FinQA accuracy ({LLM_NAME.split('/')[-1]})")]
for ax, (exp, title) in zip(axes, panels):
    part = summary[summary.experiment == exp]
    if exp == "E1 routing":
        part = part[part.method.str.endswith("[full]")]
    if part.empty:
        ax.set_axis_off(); continue
    y = np.arange(len(part))
    ax.barh(y, part.value, color=["#b06a00" if "(current)" in m else "#0c6b58" for m in part.method])
    if "ci_low" in part:
        ax.errorbar(part.value, y, xerr=[part.value - part.ci_low, part.ci_high - part.value], fmt="none", ecolor="#333", capsize=3)
    ax.set_yticks(y, [m.replace(" [full]", "") for m in part.method], fontsize=8)
    ax.set_title(title); ax.set_xlim(0, 1); ax.invert_yaxis()
plt.tight_layout(); plt.savefig(os.path.join(OUT, "summary.png"), dpi=150); plt.show()
print("Saved to", OUT)